- Delete Duplicate
- Column Schema Evolution
- DataTypes Handleing / Type casting
- Null Records Handling
- Triming
- Case Conversion
- Maintain Order of Data

In [0]:
dbutils.widgets.dropdown(name = "environment", defaultValue= "dev",choices= ["dev","prd","qa"],label = "select Environment")
env = dbutils.widgets.get("environment")
# print(env)


In [0]:
silverTablName = f"saleslake_{env}.silver_{env}.cleanedsales"
# print(silverTablName)
bronzeTablName = f"saleslake_{env}.bronze_{env}.rawsales"
# print(bronzeTablName)
srcFileLoc=f"/Volumes/saleslake_{env}/silver_{env}/vol_saleslake_src_files_{env}/daily_sales/"
# print(srcFileLoc)

In [0]:
spark.sql(f"""
INSERT INTO {silverTablName}
SELECT DISTINCT
    CAST(TRIM(sale_id)          AS INTEGER)     AS sale_id,
    CAST(TRIM(product_id)       AS INTEGER)     AS product_id,
    CAST(TRIM(store_id)         AS INTEGER)     AS store_id,
    CAST(TRIM(customer_id)      AS INTEGER)     AS customer_id,
    CAST(TRIM(region_id)        AS INTEGER)     AS region_id,
    CAST(TRIM(quantity)         AS INTEGER)     AS quantity,
    CAST(TRIM(unit_price)       AS DECIMAL(10,2)) AS unit_price,
    CAST(TRIM(gross_amount)     AS DECIMAL(10,2)) AS gross_amount,
    UPPER(TRIM(discount_code))                  AS discount_code,
    CAST(TRIM(discount_amount)  AS DECIMAL(10,2)) AS discount_amount,
    CAST(TRIM(net_amount)       AS DECIMAL(10,2)) AS net_amount,
    CAST(TRIM(cost_amount)      AS DECIMAL(10,2)) AS cost_amount,
    TO_DATE(TRIM(sale_date), 'yyyy-MM-dd')      AS sale_date,       -- e.g. 2025-06-07
    UPPER(TRIM(payment_method))                 AS payment_method,
    UPPER(TRIM(channel))                        AS channel,
    CURRENT_TIMESTAMP()                         AS ingest_ts
FROM {bronzeTablName}
WHERE ingest_ts > (
                    SELECT COALESCE(MAX(ingest_ts), TO_DATE('1990-01-01', 'yyyy-MM-dd'))
                    FROM {silverTablName}
                  )
ORDER BY CAST(TRIM(sale_id) AS INTEGER)
""")

In [0]:
%sql
SELECT count(*) FROM saleslake_dev.silver_dev.cleanedsales;

